### Usar Tracking + ReID
- Resolver problemas de oclusión, desaparición o perdida.

<img src="plots/track.png" alt="plots/track.png" width="400"/>

In [ ]:
from typing import List, Tuple
import json
from pathlib import Path

import cv2
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.colors as mcolors
from matplotlib.patches import Circle
from matplotlib.animation import FFMpegWriter
from ultralytics import YOLO
from ultralytics.engine.results import Results

from cvio.homography import Homography
from cvio.yolo import BoxYOLO


def load_points_ref(path: Path) -> List[List[int]]:
    with open(path, "r") as f:
        return json.load(f)


def crop_images_from_boxes(*, frame: np.ndarray, boxes: List[BoxYOLO]) -> List[np.ndarray]:
    """ Obtiene el recorte de una imagen a partir de la caja, para luego obtener el embedding."""
    crops = [frame[box.y1:box.y2, box.x1:box.x2] for box in boxes]
    return crops


def extract_boxes(results: List[Results]) -> List[List[BoxYOLO]]:
    boxes_yolo_all: List[List[BoxYOLO]] = []

    for res in results:
        boxes_yolo = []
        names = res.names  # dict {class_id: name}

        if res.boxes is not None:
            for box in res.boxes:
                class_id = int(box.cls.item())
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                score = float(box.conf.item()) if box.conf is not None else None
                track_id = int(box.id.item()) if box.id is not None else None

                boxes_yolo.append(BoxYOLO(
                    label=names[class_id],
                    x1=x1, y1=y1, x2=x2, y2=y2,
                    id=track_id,
                    score=score
                ))

        boxes_yolo_all.append(boxes_yolo)
    return boxes_yolo_all


DATE = "20250412T210851Z"
path_model = Path("data/yolov8x.pt")
#path_video = Path("data") / f"{DATE}.mp4"
path_video = Path("data") / "video_example.mp4"

path_points_ref = path_video.with_suffix(".json")
points_ref = load_points_ref(path_points_ref)

model = YOLO(path_model)
results = model.track(
    source=path_video,
    tracker='botsort.yaml',  # o 'strongsort.yaml'
    #show=True,               # muestra ventanas en tiempo real
    save=True,               # guarda el video con boxes
    save_txt=True,           # guarda txt con IDs por frame
    classes=[0, 15]
)

# FIXME: Hacer ReID.
boxes_yolo_all = extract_boxes(results)
for boxes_yolo in boxes_yolo_all:
    for box_yolo in boxes_yolo:
        box_yolo.reid = box_yolo.id


In [ ]:
from typing import Any, Literal

BACKGROUND_COLOR = "#86b5db"


def get_video_duration(path: Path) -> float:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise IOError(f"No se pudo abrir el video: {path}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()
    if fps == 0:
        raise ValueError(f"FPS inválido para el video: {path}")
    return frame_count / fps


def animate_simulation(
    fig: plt.Figure,
    ax: plt.Axes,
    boxes_yolo_all: List[List["BoxYOLO"]],
    tracks: dict[int, list[Tuple[float, float]]],
    lines_by_id: dict[int, plt.Line2D],
    labels_by_id: dict[int, plt.Text],
    fps: float,
    path_output: Path
) -> None:
    scatters = ax.scatter([], [], s=50, color="blue", zorder=10)

    def update(frame_idx: int):
        boxes_yolo = boxes_yolo_all[frame_idx]
        xs = [b.cx_proj for b in boxes_yolo]
        ys = [b.cy_proj for b in boxes_yolo]
        scatters.set_offsets(np.column_stack((xs, ys)))

        for box in boxes_yolo:
            if box.reid is None:
                continue
            tracks[box.reid].append((box.cx_proj, box.cy_proj))
            line_xs, line_ys = zip(*tracks[box.reid])
            lines_by_id[box.reid].set_data(line_xs, line_ys)
            labels_by_id[box.reid].set_position((box.cx_proj + 5, box.cy_proj + 5))

        return scatters,

    ani = animation.FuncAnimation(
        fig, update, frames=len(boxes_yolo_all), interval=1000 / fps, blit=True
    )

    writer = FFMpegWriter(fps=fps)
    ani.save(path_output, writer=writer)

def draw_static_elements(
    ax: plt.Axes,
    outline: list[list[float]],
    projected_refs: np.ndarray,
    padding: int = 50,
    rotation: Literal[0, 90, 180, 270] = 0
) -> None:
    outline_arr = np.array(outline, dtype=np.float32)
    all_x = np.append(outline_arr[:, 0], projected_refs[:, 0])
    all_y = np.append(outline_arr[:, 1], projected_refs[:, 1])

    min_x, max_x = np.min(all_x), np.max(all_x)
    min_y, max_y = np.min(all_y), np.max(all_y)

    ax.plot(*zip(*outline), color="black", linewidth=3)
    ax.set_aspect("equal")

    if rotation == 0:
        ax.set_xlim(min_x - padding, max_x + padding)
        ax.set_ylim(min_y - padding, max_y + padding)
    elif rotation == 90:
        ax.set_xlim(min_y - padding, max_y + padding)
        ax.set_ylim(max_x + padding, min_x - padding)
    elif rotation == 180:
        ax.set_xlim(max_x + padding, min_x - padding)
        ax.set_ylim(max_y + padding, min_y - padding)
    elif rotation == 270:
        ax.set_xlim(max_y + padding, min_y - padding)
        ax.set_ylim(min_x - padding, max_x + padding)
    else:
        raise ValueError(f"Rotación inválida: {rotation}. Debe ser 0, 90, 180 o 270.")

    for x, y in projected_refs:
        circ = Circle((x, y), radius=8, color="yellow", ec="black", zorder=5)
        ax.add_patch(circ)


def prepare_tracks(ax: plt.Axes, boxes_yolo_all: List[List["BoxYOLO"]]) -> Tuple[dict[int, list[Tuple[float, float]]], dict[int, plt.Line2D], dict[int, plt.Text]]:
    tracks: dict[int, list[Tuple[float, float]]] = {}
    lines_by_id: dict[int, plt.Line2D] = {}
    labels_by_id: dict[int, plt.Text] = {}
    colors: dict[int, Any] = {}

    cmap = matplotlib.colormaps.get_cmap("tab20")
    norm = mcolors.Normalize(vmin=0, vmax=19)

    for frame_boxes in boxes_yolo_all:
        for box in frame_boxes:
            if box.reid is None:
                continue
            if box.reid not in tracks:
                tracks[box.reid] = []
                color = cmap(norm(len(colors) % 20))
                colors[box.reid] = color
                lines_by_id[box.reid], = ax.plot([], [], color=color, linewidth=2)
                labels_by_id[box.reid] = ax.text(0, 0, str(box.reid), color=color, fontsize=9, weight="bold")

    return tracks, lines_by_id, labels_by_id


def create_simulation_animation(
    *,
    points_real: np.ndarray,
    points_ref: List[Tuple[float, float]],
    boxes_yolo_all: List[List["BoxYOLO"]],
    outline: List[List[int]],
    path_video: Path,
    path_output: Path,
    rotation: Literal[0, 90, 180, 270] = 0
) -> None:
    points_ref_np = np.array(points_ref, dtype=np.float32)
    homography = Homography(points_real=points_real, points_ref=points_ref_np)
    projected_refs = homography.project_points(points=points_ref_np)

    projected_boxes_all = []
    for boxes_yolo in boxes_yolo_all:
        pts = np.array([[box.cx, box.cy] for box in boxes_yolo], dtype=np.float32)
        if pts.size > 0:
            projected_pts = homography.project_points(points=pts)
            for box, (x, y) in zip(boxes_yolo, projected_pts):
                box.cx_proj = x
                box.cy_proj = y
        projected_boxes_all.append(boxes_yolo)

    fig, ax = plt.subplots(figsize=(8, 8))
    fig.patch.set_facecolor(BACKGROUND_COLOR)
    ax.set_facecolor(BACKGROUND_COLOR)

    draw_static_elements(ax, outline, projected_refs, rotation=rotation)
    tracks, lines_by_id, labels_by_id = prepare_tracks(ax, projected_boxes_all)

    fps = len(projected_boxes_all) / get_video_duration(path_video)
    animate_simulation(fig, ax, projected_boxes_all, tracks, lines_by_id, labels_by_id, fps, path_output)

    print(f"[INFO] Video generado: {path_output} (fps={fps:.2f})")



def get_points_real_room() -> np.ndarray:
    return np.array([
        [275, 375],
        [62.5, 273],
        [(126.5 + 62.6) / 2, 273],
        [126.5, 273],
        [62.5, (273 + 213) / 2],
        [(126.5 + 62.6) / 2, (273 + 213) / 2],
        [126.5, (273 + 213) / 2],
        [62.5, 213],
        [(126.5 + 62.6) / 2, 213],
        [126.5, 213],
    ], dtype=np.float32)


def get_points_real_yard() -> np.ndarray:
    Y_CUT = 7
    X_CUT = 18.5
    D = 20
    return np.array([
        [X_CUT, Y_CUT + D*3],
        [X_CUT, Y_CUT + D*7],
        [X_CUT, Y_CUT + D*11],
        [X_CUT, Y_CUT + D*15],
        [X_CUT, Y_CUT + D*19],
        [X_CUT + D*3, Y_CUT + D*19],
        [X_CUT + D*3, Y_CUT + D*15],
        [X_CUT + D*3, Y_CUT + D*11],
        [X_CUT + D*3, Y_CUT + D*7],
    ], dtype=np.float32)


def get_outline_room() -> List[List[float]]:
    return [
        [0, 375 + 33.5 + 50],
        [0, 375 + 33.5],
        [195, 375 + 33.5],
        [195, 375],
        [0, 375],
        [0, 0],
        [293, 0],
        [293, 375],
        [293 - 18, 375],
        [293 - 18, 375 + 33.5],
        [293 + 18.5, 375 + 33.5],
        [293 + 18.5, 375 + 33.5 + 50]
    ]


def get_outline_yard() -> List[List[float]]:
    H_TOTAL = 560
    return [
        [0, H_TOTAL],
        [0, 0],
        [156.5, 0],
        [156.5, 132],
        [156.5-55, 132],
        [156.5-55, 132+129.8],
        [156.5, 132+129.8],
        [156.5, H_TOTAL],
    ]


path_animation = path_video.with_stem(f"{path_video.stem}_animation")
points_real = get_points_real_yard()
outline = get_outline_yard()
create_simulation_animation(
    points_real=points_real,
    points_ref=points_ref,
    boxes_yolo_all=boxes_yolo_all,
    outline=outline,
    path_video=path_video,
    path_output=path_animation,
    rotation=180
)

In [ ]:
import cv2
import numpy as np

def add_points_to_video(input_video_path: str, output_video_path: str, points_ref: list):
    cap = cv2.VideoCapture(input_video_path)
    
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        for point in points_ref:
            cv2.circle(frame, tuple(point), 5, (0, 255, 255), -1)  # (0, 255, 255) es el color amarillo
        out.write(frame)

    cap.release()
    out.release()
    print(f"Video guardado en: {output_video_path}")


In [ ]:
from moviepy import VideoFileClip

def mp4_to_gif(mp4_path: str, gif_path: str) -> None:
    clip = VideoFileClip(mp4_path)
    clip.write_gif(gif_path)

path_animation_gif = path_animation.with_suffix(".gif")
mp4_to_gif(path_animation, path_animation_gif)

In [ ]:
input_video_path = Path(f"/home/pepocho/documentos/python/cvio/runs/detect/track10/{path_video.stem}.avi")
output_video_path = Path(f"/home/pepocho/documentos/python/cvio/runs/detect/track10/{path_video.stem}_with_points.avi")
output_gif_path = "asd.gif"

# Agregar puntos al video
add_points_to_video(input_video_path, output_video_path, points_ref)

# Convertir el video a GIF
mp4_to_gif(Path(output_video_path), output_gif_path)